# CS224N · Lecture 8 — Transformers

**Slides:** [cs224n-spr2024-lecture08-transformers.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture08-transformers.pdf) (Anna Goldie; adapted from slides by Anna Goldie and John Hewitt)  
**Prerequisites:** L7 (attention as queries and values), L5–6 (RNNs and their gradient problems), L3 (backprop).

### Is attention all we need?
Last lecture, attention dramatically improved recurrent seq2seq models. Today we take it one step further and ask: **is attention all we need?** *Spoiler: not quite!* A Transformer is self-attention **plus** feed-forward layers, residual connections, layer normalization, scaling, and position information. Each piece exists to fix a specific problem, and this notebook builds them one at a time, then assembles them into a **complete Transformer language model in NumPy with hand-written backprop**.

| § | Topic | You will implement / measure |
|---|---|---|
| 1 | Impact of Transformers | — |
| 2 | From recurrence to attention: complexity, path length, parallelism | Cost model; sequential vs. parallel computation |
| 3 | Self-attention: the fuzzy hashtable; the 4-step recipe | Loop and vectorized self-attention |
| 4 | The fixes: feed-forward, residuals, LayerNorm, scaling | Each one, with a measurement showing why |
| 5 | Position: sinusoids, learned, relative, 🔎 RoPE | Permutation-equivariance; RoPE's relative property |
| 6 | Multi-head attention | Heads via reshapes; parameter count |
| 7 | The decoder: causal masking, cross-attention | Masked attention; a leak test |
| 8 | **A complete Transformer LM from scratch** | Gradient-checked; trained on L5's agreement task and L6's memory task; attention maps; length extrapolation; 🔎 KV cache |
| 9 | Drawbacks and variants | Quadratic cost; sparse patterns; modern LLM block changes |
| 10 | PyTorch: a full encoder–decoder Transformer | Trained on the L6/L7 translation task |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

---
## 1. Impact of Transformers on NLP (and ML more broadly)

By the end of this lecture, you'll deeply understand the neural architecture that underpins virtually every state-of-the-art NLP model today (Vaswani et al. 2017, *Attention Is All You Need*).

* **Machine translation** (the original paper): better BLEU on WMT 2014 English–German and English–French, **and** far cheaper to train than the previous best models.
* **SuperGLUE** (a suite of challenging tasks: QA, word sense disambiguation, coreference, natural language inference): Transformer-based models went to the top of the leaderboard.
* **The rise of large language models:** Transformer-based models dominate the LMSYS Chatbot Arena leaderboard.
* **Outside NLP:** protein structure prediction (**AlphaFold2**, Jumper et al. 2021); image classification (**Vision Transformer**, Dosovitskiy et al. 2020, outperforms ResNet baselines with substantially less compute); ML for systems (a Transformer-based compiler model that speeds up a Transformer model, Zhou et al. 2020).

**Scaling laws** (Kaplan et al. 2020): with Transformers, language-modeling performance improves **smoothly as a power law** as we increase model size, training data, and compute together, over many orders of magnitude, with no sign of slowing. If we keep scaling up (with no change to the architecture), could they eventually match or exceed human-level performance? (L9–10 pick this up.)

---
## 2. From recurrence (RNNs) to attention-based NLP models

**As of last lecture,** circa 2016 the de facto strategy in NLP was: encode sentences with a **bidirectional LSTM**; define your output (parse, sentence, summary) as a sequence and use an **LSTM to generate it**; use **attention** to allow flexible access to memory.

### 2.1 Why move beyond recurrence? The Transformer authors' three desiderata
1. **Minimize (or at least not increase) computational complexity per layer.**
2. **Minimize the path length between any pair of words**, to facilitate learning long-range dependencies.
3. **Maximize the amount of computation that can be parallelized.**

Table 1 of the Transformer paper ($n$ = sequence length, $d$ = representation dimension, $k$ = kernel size):

| Layer type | Complexity per layer | Sequential operations | Maximum path length |
|---|---|---|---|
| **Self-attention** | $O(n^2\cdot d)$ | $O(1)$ | $O(1)$ |
| Recurrent | $O(n\cdot d^2)$ | $O(n)$ | $O(n)$ |
| Convolutional | $O(k\cdot n\cdot d^2)$ | $O(1)$ | $O(\log_k n)$ |

When **$n \ll d$**, complexity per layer is *lower* for a Transformer than for an RNN.

**Path length.** RNNs are unrolled left to right. This encodes **linear locality**, a useful heuristic (nearby words often affect each other's meanings). **Problem:** RNNs take $O(\text{sequence length})$ steps for distant word pairs to interact. In *"The **chef** who … **was**"*, the information about *chef* has gone through $O(n)$ layers before it meets *was*. That makes long-distance dependencies hard to learn (gradient problems!), and linear order is "baked in", while we already know sequential structure doesn't tell the whole story (L4: syntax is a tree).

**Parallelism.** Forward and backward passes of an RNN have $O(n)$ **unparallelizable** operations: future hidden states can't be computed before past ones. GPUs and TPUs can perform many independent computations at once, so this inhibits training on very large datasets, and it gets worse as sequences get longer (we can no longer batch many examples due to memory limits).

**Transformer advantages:** the number of unparallelizable operations doesn't increase with sequence length, and each "word" interacts with each other, so the maximum interaction distance is $O(1)$.

In [ ]:
# FLOP model per layer (multiply-adds): self-attention ~ n^2 d (scores + weighted sum), recurrent ~ n d^2
d = 512
for n in [32, 128, 512, 2048, 8192]:
    attn, rnn = 2 * n * n * d, n * d * d
    print(f"n={n:5d}: self-attention {attn:>14,}  recurrent {rnn:>14,}  -> {'attention cheaper' if attn < rnn else 'recurrence cheaper'}")
print(f"(crossover near n = d/2 = {d // 2} with this cost model; typical sentences are much shorter than d)")

# Sequential vs. parallel: an RNN needs n dependent steps; self-attention is a few big matrix products.
n, d_ = 256, 64
X = rng.normal(size=(n, d_)); W = rng.normal(size=(d_, d_)) / 8
t = time.perf_counter()
h = np.zeros(d_)
for i in range(n):                         # each step must wait for the previous one
    h = np.tanh(W @ h + X[i])
t_rnn = time.perf_counter() - t
t = time.perf_counter()
A = softmax(X @ X.T / np.sqrt(d_)) @ X     # all n positions at once
t_attn = time.perf_counter() - t
print(f"\nRNN over {n} steps: {1e3 * t_rnn:.2f} ms as {n} dependent steps;  self-attention: {1e3 * t_attn:.2f} ms as 1 parallel step")
print("On one CPU core either can be faster (attention does more arithmetic here). The point is the dependency structure:")
print(f"the RNN's {n} steps must run one after another, while attention's work is a few big matrix products that a GPU")
print("spreads across thousands of cores at once. That is what made Transformers so much faster to train at scale.")

---
## 3. Understanding the Transformer: self-attention

**High-level architecture: the Transformer is all about (self-)attention.** Attention treats each word's representation as a **query** to access and incorporate information from a set of **values**. Last lecture we saw attention from the decoder to the encoder. **Self-attention** is encoder–encoder (or decoder–decoder) attention, where **each word attends to each other word** within the input (or output). All words attend to all words in the previous layer.

### 3.1 Intuition: attention as a "fuzzy" or approximate hashtable
To look up a value, we compare a **query** against **keys** in a table.
* In a **hashtable**, each query (hash) maps to **exactly one** key–value pair.
* In **(self-)attention**, each query **matches each key to varying degrees**, and we return a **sum of values weighted by the query–key match**.

In [ ]:
keys = np.eye(4) * 3.0                                   # 4 keys (well separated)
values = np.array([[10., 0.], [0., 10.], [-10., 0.], [0., -10.]])
for name, q in [("query == key 1", keys[1]), ("query between keys 0 and 1", (keys[0] + keys[1]) / 2)]:
    hard = values[np.argmax(keys @ q)]                    # hashtable: exactly one value
    weights = softmax(keys @ q)                           # attention: a distribution over all values
    print(f"{name:28s} hard lookup -> {hard},  soft lookup weights {weights.round(3)} -> {weights @ values}")

### 3.2 The recipe for self-attention in the Transformer encoder
Let $w_{1:n}$ be a sequence of words, each embedded as $x_i = Ew_i\in\mathbb{R}^d$.

* **Step 1:** For each word $x_i$, calculate its **query, key, and value**, with weight matrices $Q, K, V\in\mathbb{R}^{d\times d}$:
$$q_i = Q x_i\ \ (\text{queries}),\qquad k_i = K x_i\ \ (\text{keys}),\qquad v_i = V x_i\ \ (\text{values})$$
* **Step 2:** Calculate attention scores between the query and the keys: $e_{ij} = q_i^\top k_j$
* **Step 3:** Take the softmax to normalize the attention scores: $\alpha_{ij} = \dfrac{\exp(e_{ij})}{\sum_{j'}\exp(e_{ij'})}$
* **Step 4:** Take a weighted sum of the values: $\text{output}_i = \sum_j \alpha_{ij}\,v_j$

This is exactly the **reduced-rank multiplicative attention** of L7, with $e_{ij} = (Qx_i)^\top(Kx_j)$, and with keys and values separated.

### 3.3 Vectorized
With the embeddings stacked as the rows of $X\in\mathbb{R}^{n\times d}$ (row convention, so $q_i^\top = x_i^\top Q$ etc.):
* **Step 1:** $XQ\in\mathbb{R}^{n\times d}$, $XK\in\mathbb{R}^{n\times d}$, $XV\in\mathbb{R}^{n\times d}$
* **Step 2:** all pairs of scores at once: $E = XQ(XK)^\top\in\mathbb{R}^{n\times n}$
* **Step 3:** $A = \text{softmax}(E)$, row-wise
* **Step 4:** $\text{output} = A\,XV\in\mathbb{R}^{n\times d}$

$$\text{output} = \text{softmax}\big(XQ(XK)^\top\big)\,XV$$

In [ ]:
n, d = 5, 8
X = rng.normal(size=(n, d))
Q, K, V = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))

def self_attention_loop(X, Q, K, V):
    out = []
    for i in range(len(X)):
        q = X[i] @ Q                                                # step 1 (query for word i)
        e = np.array([q @ (X[j] @ K) for j in range(len(X))])      # step 2: scores against every key
        a = softmax(e)                                              # step 3
        out.append(sum(a[j] * (X[j] @ V) for j in range(len(X))))  # step 4: weighted sum of values
    return np.array(out)

def self_attention(X, Q, K, V):
    return softmax((X @ Q) @ (X @ K).T, axis=-1) @ (X @ V)

print("loop == vectorized:", np.allclose(self_attention_loop(X, Q, K, V), self_attention(X, Q, K, V)))

---
## 4. But attention isn't quite all you need: the fixes

### 4.1 Problem: no element-wise nonlinearities → add a feed-forward layer
Since there are no element-wise non-linearities, **self-attention is simply performing a re-averaging of the value vectors**: every output is a convex combination (non-negative weights summing to 1) of the $v_j$. Stacking more self-attention layers just re-averages averages.

**Easy fix:** apply a **feed-forward network** to the output of attention, providing non-linear activation (and additional expressive power), independently at each position:
$$m_i = \text{MLP}(\text{output}_i) = W_2\,\text{ReLU}(W_1\,\text{output}_i + b_1) + b_2$$

In [ ]:
A_ = softmax((X @ Q) @ (X @ K).T)
out = A_ @ (X @ V)
print("attention weights are non-negative:", (A_ >= 0).all(), "| each row sums to 1:", np.allclose(A_.sum(1), 1))
# Every output coordinate is bounded by the min/max of the value vectors in that coordinate (convex combination):
vals = X @ V
print("outputs stay inside the values' bounding box:", ((out >= vals.min(0) - 1e-12) & (out <= vals.max(0) + 1e-12)).all())

### 4.2 How do we make this work for deep networks? Three training tricks
**Training trick #1: residual connections** (He et al. 2016). Residual connections are a simple but powerful technique from computer vision. Deep networks are surprisingly bad at learning the identity function! Directly passing "raw" embeddings to the next layer, $x_{\ell} = x_{\ell-1} + \text{Layer}(x_{\ell-1})$, prevents the network from "forgetting" or distorting important information as it is processed by many layers. Residual connections are also thought to **smooth the loss landscape** and make training easier. (L6 measured the gradient effect: the identity path passes the gradient straight through.)

**Training trick #2: layer normalization** (Ba et al. 2016). Problem: it's difficult to train the parameters of a given layer because its input from the layer beneath **keeps shifting**. Solution: reduce variation by normalizing to **zero mean and standard deviation one within each layer**, for each token's vector $x\in\mathbb{R}^d$ separately:
$$\mu = \frac1d\sum_{j=1}^d x_j,\qquad \sigma = \sqrt{\frac1d\sum_{j=1}^d(x_j-\mu)^2},\qquad \text{output} = \frac{x-\mu}{\sigma+\epsilon}\odot\gamma + \beta$$
with learned gain $\gamma\in\mathbb{R}^d$ and bias $\beta\in\mathbb{R}^d$. The original Transformer applies it after each residual addition: **"Add & Norm"**.

🔎 **LayerNorm vs. BatchNorm:** BatchNorm normalizes each feature across the *batch*, which is awkward for variable-length sequences and small batches. LayerNorm normalizes across *features* within one token, so it behaves identically at training and test time. Modern LLMs often use **RMSNorm** (no mean subtraction: $x/\text{RMS}(x)\odot\gamma$), which is slightly cheaper and works as well.

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * gamma + beta

tokens = np.array([[2.0, 4.0, 6.0, 8.0], [100.0, 0.0, -100.0, 50.0]])    # two token vectors, very different scales
ln = layer_norm(tokens, np.ones(4), np.zeros(4))
print("per-token mean:", ln.mean(-1).round(6), " per-token std:", ln.std(-1).round(4))
print(ln.round(3))

**Training trick #3: scaled dot-product attention.** After LayerNorm, the mean and variance of vector elements are 0 and 1. (Yay!) However, **the dot product still tends to take on extreme values, as its variance scales with the dimensionality $d_k$.**

Quick statistics review: for $q, k$ with independent entries of mean 0 and variance 1:
* mean of sum = sum of means: $\mathbb{E}[q^\top k] = \sum_i\mathbb{E}[q_ik_i] = 0$
* variance of sum = sum of variances: $\text{Var}(q^\top k) = \sum_{i=1}^{d_k}\text{Var}(q_ik_i) = d_k$
* **To set the variance to 1, simply divide by $\sqrt{d_k}$!**

Updated self-attention equation:
$$\text{output} = \text{softmax}\left(\frac{XQ(XK)^\top}{\sqrt{d_k}}\right)XV$$

Why it matters: large scores make the softmax nearly one-hot, and a saturated softmax has nearly **zero gradient**, so attention can't learn.

In [ ]:
r = np.random.default_rng(0)
print(f"{'d_k':>12s} {'Var(q.k)':>9s} {'mean max attention weight':>26s} {'median softmax-Jacobian norm':>29s}")
print("(averaged over 500 random queries, each scored against 16 random keys)")
for dk in [4, 64, 512]:
    var = np.var((r.normal(size=(20000, dk)) * r.normal(size=(20000, dk))).sum(1))     # empirical Var(q.k)
    for scaled in [False, True]:
        max_w, jac = [], []
        for _ in range(500):
            q, ks = r.normal(size=dk), r.normal(size=(16, dk))
            a = softmax(ks @ q / (np.sqrt(dk) if scaled else 1.0))
            max_w.append(a.max()); jac.append(np.linalg.norm(np.diag(a) - np.outer(a, a)))   # softmax Jacobian
        tag = f"{dk} (scaled)" if scaled else f"{dk}"
        print(f"{tag:>12s} {var / dk if scaled else var:9.1f} {np.mean(max_w):26.3f} {np.median(jac):29.4f}")

Without scaling, at $d_k = 512$ the scores have variance ~512, the attention is nearly one-hot (mean max weight ≈ 0.94), and the softmax Jacobian, which carries the gradient back to the scores, has collapsed to almost zero. With $1/\sqrt{d_k}$, the score variance is ~1 at every dimension, the attention stays soft, and gradients flow.

---
## 5. A major issue: word order!

Consider *"Man eats small dinosaur."* Wait a minute: **order doesn't impact the network at all!** Self-attention is **permutation-equivariant**: permuting the input words just permutes the outputs. *"Dinosaur eats small man"* gets exactly the same set of word representations. That seems wrong, given that word order does have meaning in many languages, including English!

In [ ]:
words = ["man", "eats", "small", "dinosaur"]
emb = {w: rng.normal(size=8) for w in words}
X1 = np.array([emb[w] for w in ["man", "eats", "small", "dinosaur"]])
X2 = np.array([emb[w] for w in ["dinosaur", "eats", "small", "man"]])
o1, o2 = self_attention(X1, Q, K, V), self_attention(X2, Q, K, V)
print("representation of 'man' is identical in both sentences:", np.allclose(o1[0], o2[3]))
print("representation of 'eats' is identical in both sentences:", np.allclose(o1[1], o2[1]))

### 5.1 Solution: inject order information through positional encodings
Since self-attention doesn't build in order information, we need to encode the order of the sentence in our keys, queries, and values. Represent each sequence index as a vector $p_i\in\mathbb{R}^d$, for $i\in\{1,2,\dots,n\}$. Then just **add** it to the inputs: with $\tilde v_i, \tilde k_i, \tilde q_i$ the old values, keys, and queries,
$$v_i = \tilde v_i + p_i,\qquad q_i = \tilde q_i + p_i,\qquad k_i = \tilde k_i + p_i$$
In deep self-attention networks, we do this at the **first layer** (adding $p_i$ to $x_i$). You could concatenate them as well, but people mostly just add.

### 5.2 Sinusoidal position representations (the original)
Concatenate sinusoidal functions of varying periods:
$$p_i = \begin{bmatrix}\sin(i/10000^{2\cdot1/d})\\ \cos(i/10000^{2\cdot1/d})\\ \vdots\\ \sin(i/10000^{2\cdot\frac d2/d})\\ \cos(i/10000^{2\cdot\frac d2/d})\end{bmatrix}$$
* **Pros:** periodicity indicates that maybe "absolute position" isn't as important; maybe it can extrapolate to longer sequences as periods restart.
* **Cons:** not learnable; also **the extrapolation doesn't really work** (we test this in §8.5 with learned positions).

🔎 A nice property: for every fixed offset $k$, $p_{i+k}$ is a **linear function** (a block-diagonal rotation) of $p_i$, the same for all $i$. So relative position is easy for a linear layer to extract.

In [ ]:
def sinusoidal_positions(n, d):
    pos = np.arange(n)[:, None]
    freqs = 1.0 / 10000 ** (np.arange(0, d, 2) / d)
    P = np.zeros((n, d))
    P[:, 0::2] = np.sin(pos * freqs)
    P[:, 1::2] = np.cos(pos * freqs)
    return P

P = sinusoidal_positions(100, 64)
plt.figure(figsize=(9, 3)); plt.imshow(P.T, aspect="auto", cmap="RdBu")
plt.xlabel("index in the sequence"); plt.ylabel("dimension"); plt.colorbar(); plt.title("Sinusoidal position encodings"); plt.show()

# Check the linear-shift property: P[i+k] = P[i] @ M_k for one matrix M_k, for all i.
k = 7
M_k, *_ = np.linalg.lstsq(P[:90], P[k:90 + k], rcond=None)
print(f"max error of predicting p(i+{k}) linearly from p(i), across all i: {np.abs(P[:90] @ M_k - P[k:90 + k]).max():.2e}")

### 5.3 Learned absolute positions, and relative positions
* **Learned absolute position representations:** let all $p_i$ be learnable parameters, a matrix $P\in\mathbb{R}^{d\times n}$. *Pros:* flexibility: each position gets to be learned to fit the data. *Cons:* **definitely can't extrapolate to indices outside $1,\dots,n$.** Most systems (BERT, GPT-2) used this.
* **Relative position encodings** (Shaw, Uszkoreit & Vaswani 2018). *Key insight:* the most salient position information is the **relationship** between words ("cat" is the word before "eat"), rather than their absolute position ("cat" is word 2). Relation-aware self-attention adds learned vectors that depend only on the clipped offset $j - i$:
$$e_{ij} = \frac{x_iQ\,(x_jK + a^K_{ij})^\top}{\sqrt{d_z}},\qquad z_i = \sum_j\alpha_{ij}\,(x_jV + a^V_{ij})$$
* Other ideas from the slides: **dependency-syntax-based position** (Wang et al. 2019) and **rotary embeddings** (Su et al. 2021).

### 5.4 🔎 Rotary position embeddings (RoPE), used in LLaMA, Mistral, and most modern LLMs
RoPE (Su et al. 2021) **rotates** each pair of query/key dimensions by an angle proportional to the position: for position $m$ and dimension pair $(2j, 2j+1)$, rotate by $m\theta_j$ with $\theta_j = 10000^{-2j/d}$. Because rotations compose, $\langle R_m q, R_n k\rangle = \langle q, R_{n-m}k\rangle$: **the attention score depends only on the relative offset $n - m$**, while the model still uses plain dot products. No position vectors are added to the embeddings at all.

In [ ]:
def rope(x, positions):
    # x: (n, d) with d even. Rotate each (2j, 2j+1) pair by angle position * theta_j.
    d = x.shape[-1]
    theta = 10000.0 ** (-np.arange(0, d, 2) / d)
    ang = positions[:, None] * theta[None, :]
    cos, sin = np.cos(ang), np.sin(ang)
    x1, x2 = x[:, 0::2], x[:, 1::2]
    out = np.empty_like(x)
    out[:, 0::2] = x1 * cos - x2 * sin
    out[:, 1::2] = x1 * sin + x2 * cos
    return out

q, k_ = rng.normal(size=(1, 16)), rng.normal(size=(1, 16))
for m, n_ in [(3, 5), (10, 12), (100, 102)]:          # same offset (2) at different absolute positions
    s = (rope(q, np.array([m])) @ rope(k_, np.array([n_])).T).item()
    print(f"q at position {m:3d}, k at position {n_:3d}: score {s:.6f}")
print("-> identical scores: RoPE attention depends only on the relative offset")

---
## 6. Multi-headed self-attention: $h$ heads are better than 1!

**High-level idea:** perform self-attention multiple times in parallel and combine the results.

What if we want to **look in multiple places in the sentence at once**? For word $i$, self-attention "looks" where $x_i^\top Q^\top K x_j$ is high, but maybe we want to focus on different $j$ for different reasons (e.g., one head tracks the subject, another the previous word).

* Define multiple attention "heads" through multiple $Q, K, V$ matrices: $Q_\ell, K_\ell, V_\ell\in\mathbb{R}^{d\times\frac dh}$, where $h$ is the number of heads and $\ell$ ranges from 1 to $h$.
* Each attention head performs attention independently: $\text{output}_\ell = \text{softmax}\big(XQ_\ell K_\ell^\top X^\top\big)XV_\ell$, with $\text{output}_\ell\in\mathbb{R}^{n\times d/h}$.
* Then the outputs of all the heads are combined: $\text{output} = [\text{output}_1;\dots;\text{output}_h]\,Y$, where $Y\in\mathbb{R}^{d\times d}$.
* Each head gets to "look" at different things, and construct value vectors differently.

**Efficiency:** even though we compute $h$ heads, it's not more costly. We compute $XQ\in\mathbb{R}^{n\times d}$ once, then **reshape** to $\mathbb{R}^{n\times h\times d/h}$ and transpose to $\mathbb{R}^{h\times n\times d/h}$: the heads become a batch dimension. Total parameters are the same as single-head attention ($4d^2$).

In [ ]:
def multihead_attention(X, Wq, Wk, Wv, Wo, n_heads, causal=False):
    n, d = X.shape; dk = d // n_heads
    def split(Z):                                         # (n, d) -> (h, n, d/h)
        return Z.reshape(n, n_heads, dk).transpose(1, 0, 2)
    q, k, v = split(X @ Wq), split(X @ Wk), split(X @ Wv)
    scores = q @ k.transpose(0, 2, 1) / np.sqrt(dk)        # (h, n, n): one score matrix per head
    if causal:
        scores = np.where(np.triu(np.ones((n, n), bool), 1), -np.inf, scores)
    A = softmax(scores, -1)
    heads = A @ v                                          # (h, n, d/h)
    return heads.transpose(1, 0, 2).reshape(n, d) @ Wo, A   # concatenate heads, then mix with Y = Wo

n, d, h = 6, 16, 4
X = rng.normal(size=(n, d))
Wq, Wk, Wv, Wo = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(4))
out, A = multihead_attention(X, Wq, Wk, Wv, Wo, h)

# The same thing with an explicit loop over heads, each with its own d x d/h slices:
dk = d // h
heads = []
for l in range(h):
    sl = slice(l * dk, (l + 1) * dk)
    heads.append(softmax((X @ Wq[:, sl]) @ (X @ Wk[:, sl]).T / np.sqrt(dk)) @ (X @ Wv[:, sl]))
print("reshape trick == loop over heads:", np.allclose(out, np.concatenate(heads, 1) @ Wo))
print(f"attention maps: {A.shape} (one n x n map per head);  parameters: {4 * d * d} for any number of heads")

---
## 7. The Transformer decoder

### 7.1 Masked multi-head self-attention
**Problem:** how do we keep the decoder from "cheating"? With a language-modeling objective, can't the network just look ahead and "see" the answer?

**Solution: masked multi-head attention.** At a high level, we hide (mask) information about future tokens from the model. At every timestep, we could change the set of keys and queries to include only past words, but that's inefficient. **To enable parallelization, we mask out attention to future words by setting their attention scores to $-\infty$:**
$$e_{ij} = \begin{cases}q_i^\top k_j & j\leq i\\ -\infty & j > i\end{cases}$$
(After the softmax, $e^{-\infty} = 0$: no attention to the future.) In the slides' diagram, the rows are the words being *predicted* (*The, chef, who*) and the columns are the input words they may look at (*[START], The, chef*): to predict word $i$ you may use inputs strictly before it. Indexed by input positions, as in code where the input sequence is shifted right by [START], position $i$ may attend to positions $j\leq i$, i.e. itself and everything before.

**The leak test.** If the mask works, changing a *future* input can't change any *earlier* output:

In [ ]:
X_a = rng.normal(size=(6, d)); X_b = X_a.copy(); X_b[4:] = rng.normal(size=(2, d))     # change positions 4 and 5
for causal in [False, True]:
    o_a, _ = multihead_attention(X_a, Wq, Wk, Wv, Wo, h, causal=causal)
    o_b, _ = multihead_attention(X_b, Wq, Wk, Wv, Wo, h, causal=causal)
    print(f"causal={causal}: outputs at positions 0-3 unchanged after editing positions 4-5? {np.allclose(o_a[:4], o_b[:4])}")
_, A = multihead_attention(X_a, Wq, Wk, Wv, Wo, h, causal=True)
print("head 0 attention (lower-triangular):\n", A[0].round(2))

### 7.2 Encoder–decoder (cross-)attention
* Self-attention is when keys, queries, and values come from the **same** source.
* In the decoder, we also have attention that looks more like what we saw last week. Let $h_1,\dots,h_n$ be **output vectors from the Transformer encoder** and $z_1,\dots,z_n$ be **input vectors from the Transformer decoder**.
* Then **keys and values are drawn from the encoder** (like a memory): $k_i = Kh_i$, $v_i = Vh_i$.
* And **queries are drawn from the decoder**: $q_i = Qz_i$.

### 7.3 Finishing touches, and the full architecture
* Add a **feed-forward layer** (with residual connections and layer norm).
* Add a **final linear layer** to project the embeddings into a much longer vector of length vocab size (**logits**).
* Add a **final softmax** to generate a probability distribution of possible next words!

**Recap of the Transformer (Vaswani et al. 2017):**
* **Encoder** (repeat ×6): input embedding + positional encoding → [multi-head self-attention → Add & Norm → feed-forward → Add & Norm]
* **Decoder** (repeat ×6): output embedding (outputs shifted right) + positional encoding → [**masked** multi-head self-attention → Add & Norm → multi-head **cross**-attention over the encoder outputs → Add & Norm → feed-forward → Add & Norm] → linear → softmax → output probabilities

🔎 **Post-LN vs. pre-LN.** The original "Add & Norm" is **post-LN**: $x \leftarrow \text{LN}(x + \text{Sublayer}(x))$. Almost all modern models use **pre-LN**: $x \leftarrow x + \text{Sublayer}(\text{LN}(x))$, plus a final LN before the output layer. Pre-LN keeps a clean identity path from input to output, so deep models train stably without careful learning-rate warmup (Xiong et al. 2020). Our implementation below uses pre-LN.

🔎 **Three families of Transformers** (L9): **encoder-only** (BERT: bidirectional self-attention, for understanding tasks), **decoder-only** (GPT, Claude, LLaMA: causal self-attention, a language model), and **encoder–decoder** (the original, T5: for seq2seq tasks like MT).

---
## 8. A complete Transformer language model, from scratch

A **decoder-only** Transformer (the GPT architecture), in NumPy, with every gradient written by hand:
$$x^{(0)}_i = E_{w_i} + P_i\qquad\text{(token + learned absolute position embeddings)}$$
$$\text{for each layer: } \quad a = x + \text{MHA}_{\text{causal}}(\text{LN}_1(x)),\qquad x \leftarrow a + \text{FFN}(\text{LN}_2(a))$$
$$\text{logits}_i = \text{LN}_f(x_i)\,W_{\text{out}},\qquad J = -\frac1T\sum_i\log\text{softmax}(\text{logits}_i)_{w_{i+1}}$$

The backward pass uses only pieces you've already derived: softmax+CE ($\hat y - y$, L3), linear layers ($\delta x^\top$), ReLU (routing), residuals (the gradient **adds** along both paths), LayerNorm, and attention (softmax backward, as in L7). The new piece is LayerNorm's backward. With $\hat x = (x-\mu)/\sigma$ and upstream gradient $g = \gamma\odot\partial J/\partial y$:
$$\frac{\partial J}{\partial x} = \frac{1}{\sigma}\Big(g - \text{mean}(g) - \hat x\odot\text{mean}(g\odot\hat x)\Big)$$

In [ ]:
def ln_forward(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    xh = (x - mu) / np.sqrt(var + eps)
    return xh * g + b, (xh, var, eps)

def ln_backward(dy, g, cache):
    xh, var, eps = cache; D = xh.shape[-1]
    dg = (dy * xh).reshape(-1, D).sum(0); db = dy.reshape(-1, D).sum(0)
    dxh = dy * g
    dx = (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True)) / np.sqrt(var + eps)
    return dx, dg, db

def init_transformer(V, T_max, d=64, n_layers=2, n_heads=4, d_ff=128, seed=0):
    r = np.random.default_rng(seed)
    p = {"tok": r.normal(0, 0.02, (V, d)), "pos": r.normal(0, 0.02, (T_max, d))}
    for l in range(n_layers):
        for k in ["Wq", "Wk", "Wv", "Wo"]:
            p[f"{l}.{k}"] = r.normal(0, 1 / np.sqrt(d), (d, d))
        p[f"{l}.W1"], p[f"{l}.b1"] = r.normal(0, 1 / np.sqrt(d), (d, d_ff)), np.zeros(d_ff)
        p[f"{l}.W2"], p[f"{l}.b2"] = r.normal(0, 1 / np.sqrt(d_ff), (d_ff, d)), np.zeros(d)
        for k in ["ln1", "ln2"]:
            p[f"{l}.{k}g"], p[f"{l}.{k}b"] = np.ones(d), np.zeros(d)
    p["lnfg"], p["lnfb"] = np.ones(d), np.zeros(d)
    p["Wout"] = r.normal(0, 1 / np.sqrt(d), (d, V))
    return p

def mha_forward(a, Wq, Wk, Wv, Wo, nh):
    B, T, d = a.shape; dk = d // nh
    split = lambda z: z.reshape(B, T, nh, dk).transpose(0, 2, 1, 3)       # (B, h, T, dk)
    q, k, v = split(a @ Wq), split(a @ Wk), split(a @ Wv)
    S = q @ k.transpose(0, 1, 3, 2) / np.sqrt(dk)
    S = np.where(np.triu(np.ones((T, T), bool), 1), -1e9, S)              # causal mask
    P = softmax(S, -1)
    Zc = (P @ v).transpose(0, 2, 1, 3).reshape(B, T, d)                    # concatenate heads
    return Zc @ Wo, (a, q, k, v, P, Zc, dk)

def mha_backward(dout, Wq, Wk, Wv, Wo, nh, cache):
    a, q, k, v, P, Zc, dk = cache; B, T, d = a.shape
    gWo = Zc.reshape(-1, d).T @ dout.reshape(-1, d)
    dZ = (dout @ Wo.T).reshape(B, T, nh, dk).transpose(0, 2, 1, 3)
    dP = dZ @ v.transpose(0, 1, 3, 2); dv = P.transpose(0, 1, 3, 2) @ dZ
    dS = P * (dP - (dP * P).sum(-1, keepdims=True)) / np.sqrt(dk)          # softmax backward (+ the 1/sqrt(dk))
    dq, dk_ = dS @ k, dS.transpose(0, 1, 3, 2) @ q
    merge = lambda z: z.transpose(0, 2, 1, 3).reshape(B, T, d)
    dq, dk_, dv = merge(dq), merge(dk_), merge(dv)
    a2 = a.reshape(-1, d)
    grads = (a2.T @ dq.reshape(-1, d), a2.T @ dk_.reshape(-1, d), a2.T @ dv.reshape(-1, d), gWo)
    return dq @ Wq.T + dk_ @ Wk.T + dv @ Wv.T, grads

def transformer_lm(p, X, Y, M, n_layers, n_heads, need_grad=True, return_attention=False):
    B, T = X.shape
    x = p["tok"][X] + p["pos"][:T]
    caches, attn_maps = [], []
    for l in range(n_layers):
        a_in, c1 = ln_forward(x, p[f"{l}.ln1g"], p[f"{l}.ln1b"])
        att, ca = mha_forward(a_in, p[f"{l}.Wq"], p[f"{l}.Wk"], p[f"{l}.Wv"], p[f"{l}.Wo"], n_heads)
        attn_maps.append(ca[4])
        x1 = x + att                                                          # residual 1
        f_in, c2 = ln_forward(x1, p[f"{l}.ln2g"], p[f"{l}.ln2b"])
        hpre = f_in @ p[f"{l}.W1"] + p[f"{l}.b1"]; hid = np.maximum(hpre, 0)
        x = x1 + hid @ p[f"{l}.W2"] + p[f"{l}.b2"]                            # residual 2
        caches.append((c1, ca, c2, f_in, hpre, hid))
    xf, cf = ln_forward(x, p["lnfg"], p["lnfb"])
    Pr = softmax(xf @ p["Wout"], -1)
    Bi, Ti = np.meshgrid(np.arange(B), np.arange(T), indexing="ij"); ntok = M.sum()
    loss = -(np.log(Pr[Bi, Ti, Y] + 1e-12) * M).sum() / ntok
    if not need_grad:
        return (loss, Pr, attn_maps) if return_attention else (loss, Pr)
    g = {}; d = x.shape[-1]
    G = Pr.copy(); G[Bi, Ti, Y] -= 1; G *= M[:, :, None] / ntok
    g["Wout"] = xf.reshape(-1, d).T @ G.reshape(-1, G.shape[-1])
    dx, g["lnfg"], g["lnfb"] = ln_backward(G @ p["Wout"].T, p["lnfg"], cf)
    for l in reversed(range(n_layers)):
        c1, ca, c2, f_in, hpre, hid = caches[l]
        g[f"{l}.W2"] = hid.reshape(-1, hid.shape[-1]).T @ dx.reshape(-1, d); g[f"{l}.b2"] = dx.reshape(-1, d).sum(0)
        dhpre = (dx @ p[f"{l}.W2"].T) * (hpre > 0)
        g[f"{l}.W1"] = f_in.reshape(-1, d).T @ dhpre.reshape(-1, dhpre.shape[-1]); g[f"{l}.b1"] = dhpre.reshape(-1, dhpre.shape[-1]).sum(0)
        dx1_ln, g[f"{l}.ln2g"], g[f"{l}.ln2b"] = ln_backward(dhpre @ p[f"{l}.W1"].T, p[f"{l}.ln2g"], c2)
        dx1 = dx + dx1_ln                                                     # residual: gradients add
        da, (g[f"{l}.Wq"], g[f"{l}.Wk"], g[f"{l}.Wv"], g[f"{l}.Wo"]) = mha_backward(
            dx1, p[f"{l}.Wq"], p[f"{l}.Wk"], p[f"{l}.Wv"], p[f"{l}.Wo"], n_heads, ca)
        dxa, g[f"{l}.ln1g"], g[f"{l}.ln1b"] = ln_backward(da, p[f"{l}.ln1g"], c1)
        dx = dx1 + dxa
    g["tok"] = np.zeros_like(p["tok"]); np.add.at(g["tok"], X, dx)
    g["pos"] = np.zeros_like(p["pos"]); g["pos"][:T] = dx.sum(0)
    return loss, g

**Gradient check of every parameter tensor** on a tiny model. We first perturb the parameters away from the initialization (where LayerNorm gains are exactly 1 and biases 0), so that every gradient is non-trivial.

In [ ]:
r = np.random.default_rng(0)
Vt, Tt = 7, 5
p_chk = init_transformer(Vt, 8, d=8, n_layers=2, n_heads=2, d_ff=12, seed=1)
for k in p_chk:
    p_chk[k] = p_chk[k] + r.normal(0, 0.3, p_chk[k].shape)
Xc, Yc = r.integers(0, Vt, (3, Tt)), r.integers(0, Vt, (3, Tt)); Mc = (r.random((3, Tt)) > 0.2).astype(float)
_, g_chk = transformer_lm(p_chk, Xc, Yc, Mc, 2, 2)
worst = {}
for k in p_chk:
    num = np.zeros_like(p_chk[k])
    for idx in np.ndindex(p_chk[k].shape):
        old = p_chk[k][idx]
        p_chk[k][idx] = old + 1e-6; lp = transformer_lm(p_chk, Xc, Yc, Mc, 2, 2, False)[0]
        p_chk[k][idx] = old - 1e-6; lm_ = transformer_lm(p_chk, Xc, Yc, Mc, 2, 2, False)[0]
        p_chk[k][idx] = old
        num[idx] = (lp - lm_) / 2e-6
    worst[k] = np.abs(num - g_chk[k]).max()
print(f"checked {len(worst)} parameter tensors; worst max-abs error = {max(worst.values()):.1e} (in {max(worst, key=worst.get)})")

### 8.1 Training on the long-distance agreement task (from L5)
The same artificial language as Lecture 5: the verb must agree with a subject that can be many words back, across attractor nouns. We use Adam, gradient clipping, a 2-layer, 4-head model with $d = 64$.

In [ ]:
NOUNS = [("dog", "dogs"), ("cat", "cats"), ("pilot", "pilots"), ("farmer", "farmers"), ("author", "authors"), ("key", "keys")]
VERBS = [("sleeps", "sleep"), ("laughs", "laugh"), ("waits", "wait"), ("smiles", "smile")]
TRANS = ["saw", "liked", "chased", "met", "helped"]
ADJ = ["old", "young", "tall", "small", "happy"]
PREP = ["near", "behind", "with"]

def gen_sentence(max_mods=3, rng=random):
    plural = rng.random() < 0.5
    w = ["the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else [])
    w.append(rng.choice(NOUNS)[plural]); subj = len(w) - 1
    for _ in range(rng.choice(range(max_mods + 1))):
        ap = rng.random() < 0.5
        if rng.random() < 0.5:
            w += ["that", "the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else []) + [rng.choice(NOUNS)[ap], rng.choice(TRANS)]
        else:
            w += [rng.choice(PREP), "the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else []) + [rng.choice(NOUNS)[ap]]
    vpair = rng.choice(VERBS); w.append(vpair[plural]); verb = len(w) - 1
    if rng.random() < 0.3:
        w.append("quietly")
    w.append(".")
    return w, subj, verb, plural, vpair

random.seed(0)
train_data = [gen_sentence() for _ in range(6000)]
test_data = [gen_sentence() for _ in range(1000)]
test_long = [gen_sentence(max_mods=8) for _ in range(1500)]          # much longer sentences, for Sec. 8.5
vocab = ["<s>", "</s>"] + sorted({w for s, *_ in train_data + test_long for w in s})
w2i = {w: i for i, w in enumerate(vocab)}; V = len(vocab)
T_TRAIN = max(len(s) for s, *_ in train_data) + 1
T_MAX = max(len(s) for s, *_ in test_long) + 1
print(f"|V| = {V}; longest training input = {T_TRAIN} tokens; longest long-test input = {T_MAX} tokens")

def batchify(sents):
    L = max(len(s[0]) for s in sents) + 1
    X = np.zeros((len(sents), L), int); Y = np.zeros((len(sents), L), int); M = np.zeros((len(sents), L))
    for b, (s, *_) in enumerate(sents):
        ids = [w2i["<s>"]] + [w2i[w] for w in s] + [w2i["</s>"]]
        X[b, :len(ids) - 1] = ids[:-1]; Y[b, :len(ids) - 1] = ids[1:]; M[b, :len(ids) - 1] = 1
    return X, Y, M

class Adam:
    def __init__(self, params, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            params[k] -= self.lr * (self.m[k] / (1 - self.b1 ** self.t)) / (np.sqrt(self.v[k] / (1 - self.b2 ** self.t)) + self.eps)

def clip_grads(g, max_norm=1.0):
    norm = math.sqrt(sum((v ** 2).sum() for v in g.values()))
    if norm > max_norm:
        for k in g:
            g[k] *= max_norm / norm

N_LAYERS, N_HEADS = 2, 4
def bucket(d):
    return "1" if d == 1 else ("2-6" if d <= 6 else ("7-10" if d <= 10 else "11+"))

def agreement(p, sents, by="distance"):
    acc = collections.defaultdict(list)
    for i in range(0, len(sents), 100):
        chunk = sents[i:i + 100]; X, Y, M = batchify(chunk)
        _, P = transformer_lm(p, X, Y, M, N_LAYERS, N_HEADS, False)
        for b, (s, sp, vp, pl, vpair) in enumerate(chunk):
            key = bucket(vp - sp) if by == "distance" else ("verb within trained positions" if vp < T_TRAIN else "verb beyond trained positions")
            acc[key].append(P[b, vp, w2i[vpair[pl]]] > P[b, vp, w2i[vpair[1 - pl]]])
    return {k: (round(float(np.mean(v)), 3), len(v)) for k, v in sorted(acc.items())}

p_tf = init_transformer(V, T_MAX, d=64, n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=128)
opt = Adam(p_tf, lr=1e-3); r = np.random.default_rng(0); t0 = time.time()
STEPS = 2000
for step in range(1, STEPS + 1):
    X, Y, M = batchify([train_data[i] for i in r.integers(0, len(train_data), 64)])
    loss, g = transformer_lm(p_tf, X, Y, M, N_LAYERS, N_HEADS)
    clip_grads(g); opt.step(p_tf, g)
    if step % 500 == 0:
        Xt, Yt, Mt = batchify(test_data)
        ppl = math.exp(transformer_lm(p_tf, Xt, Yt, Mt, N_LAYERS, N_HEADS, False)[0])
        print(f"step {step}: train loss {loss:.4f}  test perplexity {ppl:.3f}  agreement {({k: v[0] for k, v in agreement(p_tf, test_data).items()})}  ({time.time() - t0:.0f}s)")

The Transformer reaches the same near-perfect agreement as L5's RNN, and a similar perplexity (L5's RNN: ≈3.98). Notice the test perplexity is best early and then creeps up while the training loss keeps falling: with no dropout, the model starts to **overfit** the 6,000 training sentences. (L7's advice: regularize until it doesn't overfit on dev data; Exercise 2 adds dropout.) But it gets there differently from the RNN: no information has to be carried step by step. At the verb, it can **attend directly to the subject**.

### 8.2 What are the heads looking at?
For a test sentence, we plot each head's attention from the position that predicts the verb (the word right before it) back over the sentence.

In [ ]:
s, sp, vp, pl, vpair = next(x for x in test_data if x[2] - x[1] >= 8)
X, Y, M = batchify([(s, sp, vp, pl, vpair)])
_, _, maps = transformer_lm(p_tf, X, Y, M, N_LAYERS, N_HEADS, False, return_attention=True)
inputs = ["<s>"] + s
fig, axes = plt.subplots(N_LAYERS, 1, figsize=(10, 2.2 * N_LAYERS))
for l in range(N_LAYERS):
    w = maps[l][0, :, vp, :vp + 1]                       # (heads, positions <= vp): query = the input predicting the verb
    axes[l].imshow(w, cmap="Greys", aspect="auto", vmin=0, vmax=1)
    axes[l].set_yticks(range(N_HEADS)); axes[l].set_yticklabels([f"L{l} head {h_}" for h_ in range(N_HEADS)])
    axes[l].set_xticks(range(vp + 1)); axes[l].set_xticklabels(inputs[:vp + 1], rotation=50, ha="right")
plt.suptitle(f"Attention from the position that predicts '{s[vp]}' (subject = '{s[sp]}')"); plt.tight_layout(); plt.show()
subj_input_pos = sp + 1                                   # inputs are shifted by <s>
for l in range(N_LAYERS):
    for h_ in range(N_HEADS):
        top = maps[l][0, h_, vp, :vp + 1].argmax()
        print(f"layer {l} head {h_}: strongest attention on '{inputs[top]}' (weight {maps[l][0, h_, vp, top]:.2f})"
              + ("  <- the subject" if top == subj_input_pos else ""))

(Head roles vary between training runs, and they are not always cleanly interpretable; attention weights are a hint, not a full explanation. Typically you'll find at least one head that focuses on the subject noun at the verb position, i.e., a head that has learned a **syntactic** relation, without ever being told about syntax.)

### 8.3 The long-gap memory task (from L6)
In L6, remembering one symbol across a 60-step gap was hard for a vanilla RNN, and possible for an LSTM only with the right forget-gate bias. For self-attention, the first token is **one step away** from the last, at any distance.

In [ ]:
V_MEM = 12
def memory_batch(T, n, r):
    X = r.integers(4, 11, size=(n, T + 2)); s = r.integers(0, 4, size=n); X[:, 0] = s; X[:, -1] = 11
    Y = np.zeros_like(X); Y[:, -1] = s; M = np.zeros(X.shape); M[:, -1] = 1
    return X, Y, M

for T_GAP in [60]:
    p_m = init_transformer(V_MEM, T_GAP + 2, d=32, n_layers=2, n_heads=2, d_ff=64, seed=0)
    opt_m = Adam(p_m, lr=3e-3); r = np.random.default_rng(0); t0 = time.time()
    for step in range(1, 601):
        X, Y, M = memory_batch(T_GAP, 64, r)
        _, g = transformer_lm(p_m, X, Y, M, 2, 2); clip_grads(g); opt_m.step(p_m, g)
        if step % 150 == 0:
            X, Y, M = memory_batch(T_GAP, 500, np.random.default_rng(99))
            _, P = transformer_lm(p_m, X, Y, M, 2, 2, False)
            print(f"gap {T_GAP}, step {step}: accuracy {(P[:, -1].argmax(-1) == Y[:, -1]).mean():.2f}  ({time.time() - t0:.0f}s)")

### 8.4 🔎 Fast generation with a KV cache
At generation time, a decoder produces one token at a time. Recomputing attention over the whole prefix at every step costs $O(t)$ projections per step for $t$ steps. But because of causal masking, **the keys and values of earlier positions never change**, so we can **cache** them and only compute the new token's query, key, and value. Every LLM inference server does this; the cache's memory ($2\times\text{layers}\times t\times d$ numbers per sequence) is often the bottleneck of serving, which motivates multi-query / grouped-query attention (§9).

We verify that cached incremental decoding gives exactly the same next-token distributions as running the full model on the prefix.

In [ ]:
def step_with_cache(p, token, pos, cache, n_layers, n_heads):
    # Process ONE new token at position `pos`, reusing cached keys/values. Returns next-token probabilities.
    x = (p["tok"][token] + p["pos"][pos])[None, :]                       # (1, d)
    d = x.shape[-1]; dk = d // n_heads
    for l in range(n_layers):
        a, _ = ln_forward(x, p[f"{l}.ln1g"], p[f"{l}.ln1b"])
        q, k, v = a @ p[f"{l}.Wq"], a @ p[f"{l}.Wk"], a @ p[f"{l}.Wv"]
        cache[l]["k"].append(k[0]); cache[l]["v"].append(v[0])
        K_, V_ = np.array(cache[l]["k"]), np.array(cache[l]["v"])          # (t, d): all keys/values so far
        qh = q.reshape(n_heads, dk); Kh = K_.reshape(-1, n_heads, dk).transpose(1, 0, 2); Vh = V_.reshape(-1, n_heads, dk).transpose(1, 0, 2)
        att = softmax(np.einsum("hd,htd->ht", qh, Kh) / np.sqrt(dk), -1)  # no mask needed: the cache holds only the past
        z = np.einsum("ht,htd->hd", att, Vh).reshape(1, d) @ p[f"{l}.Wo"]
        x = x + z
        f_in, _ = ln_forward(x, p[f"{l}.ln2g"], p[f"{l}.ln2b"])
        x = x + np.maximum(f_in @ p[f"{l}.W1"] + p[f"{l}.b1"], 0) @ p[f"{l}.W2"] + p[f"{l}.b2"]
    xf, _ = ln_forward(x, p["lnfg"], p["lnfb"])
    return softmax(xf @ p["Wout"])[0]

prefix = ["<s>"] + "the dogs that the cat saw near the old farmer".split()
ids = [w2i[w] for w in prefix]
cache = [{"k": [], "v": []} for _ in range(N_LAYERS)]
max_diff = 0.0
for t, tok in enumerate(ids):
    p_cached = step_with_cache(p_tf, tok, t, cache, N_LAYERS, N_HEADS)
    X = np.array([ids[:t + 1]])
    _, P_full = transformer_lm(p_tf, X, X, np.ones(X.shape), N_LAYERS, N_HEADS, False)
    max_diff = max(max_diff, np.abs(p_cached - P_full[0, -1]).max())
print(f"max difference between cached and full recomputation: {max_diff:.1e}")
top = np.argsort(-p_cached)[:4]
print("next-word prediction after '" + " ".join(prefix[1:]) + "':", [(vocab[i], round(float(p_cached[i]), 3)) for i in top])

### 8.5 Learned positions don't extrapolate
Our model uses **learned absolute position embeddings**, and it was trained only on inputs of length ≤ `T_TRAIN`. Embedding rows beyond that were never updated: they're still random initialization. What happens on much longer sentences (up to `T_MAX` tokens)?

In [ ]:
res = agreement(p_tf, test_long, by="position")
for k, (acc, n_) in res.items():
    print(f"{k:32s}: agreement accuracy {acc:.3f}  ({n_} sentences)")
Xl, Yl, Ml = batchify(test_long[:300])
_, Pl = transformer_lm(p_tf, Xl, Yl, Ml, N_LAYERS, N_HEADS, False)
nll = -np.log(Pl[np.arange(len(Xl))[:, None], np.arange(Xl.shape[1])[None], Yl] + 1e-12) * Ml
per_pos = nll.sum(0) / np.maximum(Ml.sum(0), 1)
plt.figure(figsize=(8, 3.5))
plt.plot(per_pos[:T_MAX], "o-", ms=3); plt.axvline(T_TRAIN - 0.5, color="red", ls="--", label="longest training input")
plt.xlabel("position"); plt.ylabel("mean loss (nats)"); plt.title("Loss by position on longer-than-training sentences"); plt.legend(); plt.show()
w_in, w_out = Ml[:, :T_TRAIN].sum(), Ml[:, T_TRAIN:].sum()
print(f"mean loss at positions < {T_TRAIN}: {nll[:, :T_TRAIN].sum() / w_in:.3f} nats;  at positions >= {T_TRAIN}: {nll[:, T_TRAIN:].sum() / w_out:.3f} nats")

Inside the trained range, agreement stays high on these longer, more attractor-laden sentences. When the verb falls **past the longest training length**, accuracy drops markedly, and the per-position loss rises beyond the red line. Those position embeddings are still their random initialization: the model is reading positions it never saw. (By contrast, in our runs a vanilla RNN trained on the same short sentences kept ~98–100% agreement on the long ones. It has no notion of absolute position, only a recurrence that it applies again and again; Exercise 7 asks you to reproduce this with L5's code.) This is the "can't extrapolate" con from §5.3, measured. It's a major reason relative schemes like **RoPE** and **ALiBi** (Press et al. 2022) replaced learned absolute positions, and why "context-length extension" of LLMs is an active research area.

---
## 9. Drawbacks and variants of Transformers

What would we like to fix about the Transformer?
* **Quadratic compute in self-attention:** computing all pairs of interactions means computation grows **quadratically** with sequence length (for recurrent models it only grew linearly).
* **Position representations:** are simple absolute indices the best we can do? Alternatives: relative linear position attention (Shaw et al. 2018), dependency-syntax-based position (Wang et al. 2019), rotary embeddings (Su et al. 2021).

In [ ]:
print(f"{'seq len':>8s} {'attention scores per head per layer':>36s} {'memory (fp16), 32 heads x 32 layers':>38s}")
for n in [1_024, 8_192, 32_768, 131_072]:
    scores = n * n
    gb = scores * 2 * 32 * 32 / 1e9
    print(f"{n:8,d} {scores:36,d} {gb:35,.0f} GB")
print("(if you naively materialized every score matrix; FlashAttention avoids storing them at all)")

### 9.1 Recent work on improving on the quadratic self-attention cost
Considerable work has gone into the question: **can we build models like Transformers without paying the $O(T^2)$ all-pairs self-attention cost?**
* **Linformer** (Wang et al. 2020). *Key idea:* map the **sequence-length dimension** to a lower-dimensional space for values and keys: project the $n\times d$ keys and values to $k\times d$ with learned $k\times n$ matrices, so attention costs $O(nk)$. Its inference time stays nearly flat as sequence length grows.
* **BigBird** (Zaheer et al. 2021). *Key idea:* replace all-pairs interactions with a family of other interactions, like **local windows**, looking at everything (**global** tokens), and **random** interactions.

In [ ]:
def bigbird_mask(n, window=2, n_global=2, n_random=2, seed=0):
    r = np.random.default_rng(seed); M = np.zeros((n, n), bool)
    for i in range(n):
        M[i, max(0, i - window):i + window + 1] = True          # local window
        M[i, r.choice(n, n_random, replace=False)] = True       # random connections
    M[:n_global, :] = True; M[:, :n_global] = True              # global tokens see / are seen by everything
    return M

n = 32
masks = {"full attention": np.ones((n, n), bool), "local window": bigbird_mask(n, n_global=0, n_random=0),
         "BigBird (local + global + random)": bigbird_mask(n)}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, Mk) in zip(axes, masks.items()):
    ax.imshow(Mk, cmap="Blues"); ax.set_title(f"{name}\n{Mk.sum()} of {n * n} pairs"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# Linformer's idea in two lines: project the n keys/values down to k "summary" rows.
n, d, k = 512, 64, 32
Xl = rng.normal(size=(n, d)); E_proj = rng.normal(size=(k, n)) / np.sqrt(n)
scores_full = Xl @ Xl.T                                             # (n, n)
scores_lin = Xl @ (E_proj @ Xl).T                                   # (n, k)
print(f"Linformer score matrix {scores_lin.shape} instead of {scores_full.shape}")

### 9.2 Do Transformer modifications transfer?
Narang et al. (2021) re-implemented dozens of proposed Transformer modifications in one codebase and found: **"Surprisingly, we find that most modifications do not meaningfully improve performance."** The vanilla Transformer is a very strong baseline; be skeptical of small architectural tweaks.

### 9.3 🔎 What actually changed between the 2017 Transformer and a 2024 LLM block
A handful of modifications *did* stick. If you read the LLaMA or Mistral papers, here's the dictionary:

| Component | Original (2017) | Typical modern LLM | Why |
|---|---|---|---|
| Normalization placement | post-LN ("Add & Norm") | **pre-LN** | stable training of deep models |
| Normalization | LayerNorm | **RMSNorm** | cheaper, works as well |
| Positions | sinusoidal, added to inputs | **RoPE**, applied to q and k | relative positions, better long-context behavior |
| FFN | ReLU, $4d$ hidden | **SwiGLU**: $(\text{Swish}(xW_1)\odot xW_3)W_2$ | consistently better quality per FLOP |
| Attention heads | full multi-head | **grouped-query attention** (several query heads share one K/V head) | smaller KV cache, faster inference |
| Attention kernel | materialize $n\times n$ | **FlashAttention** (Dao et al. 2022) | exact attention, tiled, IO-aware: much faster, linear memory |
| Biases | everywhere | often **none** in linear layers | simplicity, stability |
| Architecture | encoder–decoder | **decoder-only** | one model for LM pretraining and generation |

---
## 10. PyTorch: a full encoder–decoder Transformer, from building blocks

Everything from §3–7 in PyTorch: multi-head attention (self, causal, cross), sinusoidal positions, post-LN "Add & Norm" exactly as in the paper's diagram, trained on the toy translation task from L6–7. Compare its accuracy with L7's LSTM + attention model. We check our attention against PyTorch's fused `F.scaled_dot_product_attention` first.

In [ ]:
# [PyTorch] Encoder-decoder Transformer from scratch on the toy MT task.
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h, self.dk = n_heads, d // n_heads
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))
    def forward(self, x_q, x_kv, key_padding=None, causal=False):
        B, Tq, d = x_q.shape; Tk = x_kv.shape[1]
        q = self.Wq(x_q).view(B, Tq, self.h, self.dk).transpose(1, 2)            # (B, h, Tq, dk)
        k = self.Wk(x_kv).view(B, Tk, self.h, self.dk).transpose(1, 2)
        v = self.Wv(x_kv).view(B, Tk, self.h, self.dk).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)                    # scaled dot product
        if key_padding is not None:                                             # (B, Tk) True = padding
            scores = scores.masked_fill(key_padding[:, None, None, :], float("-inf"))
        if causal:
            scores = scores.masked_fill(torch.triu(torch.ones(Tq, Tk, dtype=torch.bool), 1), float("-inf"))
        out = scores.softmax(-1) @ v
        return self.Wo(out.transpose(1, 2).reshape(B, Tq, d))

# sanity check against PyTorch's fused kernel
mha = MultiHeadAttention(16, 4); xx = torch.randn(2, 5, 16)
q = mha.Wq(xx).view(2, 5, 4, 4).transpose(1, 2); k = mha.Wk(xx).view(2, 5, 4, 4).transpose(1, 2); v = mha.Wv(xx).view(2, 5, 4, 4).transpose(1, 2)
ref = mha.Wo(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(2, 5, 16))
print("matches F.scaled_dot_product_attention:", torch.allclose(mha(xx, xx, causal=True), ref, atol=1e-6))

class EncoderLayer(nn.Module):
    def __init__(self, d, h, d_ff, drop=0.1):
        super().__init__()
        self.att, self.ff = MultiHeadAttention(d, h), nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
        self.n1, self.n2, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(drop)
    def forward(self, x, pad):
        x = self.n1(x + self.drop(self.att(x, x, key_padding=pad)))       # Add & Norm
        return self.n2(x + self.drop(self.ff(x)))                         # Add & Norm

class DecoderLayer(nn.Module):
    def __init__(self, d, h, d_ff, drop=0.1):
        super().__init__()
        self.self_att, self.cross_att = MultiHeadAttention(d, h), MultiHeadAttention(d, h)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
        self.n1, self.n2, self.n3, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(drop)
    def forward(self, z, enc, src_pad):
        z = self.n1(z + self.drop(self.self_att(z, z, causal=True)))                 # masked self-attention
        z = self.n2(z + self.drop(self.cross_att(z, enc, key_padding=src_pad)))     # queries: decoder; keys/values: encoder
        return self.n3(z + self.drop(self.ff(z)))

class Transformer(nn.Module):
    def __init__(self, n_src, n_tgt, d=64, h=4, d_ff=128, layers=2, max_len=64):
        super().__init__()
        self.d = d
        self.src_emb, self.tgt_emb = nn.Embedding(n_src, d), nn.Embedding(n_tgt, d)
        self.register_buffer("pos", torch.tensor(sinusoidal_positions(max_len, d), dtype=torch.float32))
        self.enc = nn.ModuleList(EncoderLayer(d, h, d_ff) for _ in range(layers))
        self.dec = nn.ModuleList(DecoderLayer(d, h, d_ff) for _ in range(layers))
        self.out = nn.Linear(d, n_tgt)
    def encode(self, src):
        pad = src == 0
        x = self.src_emb(src) * math.sqrt(self.d) + self.pos[:src.shape[1]]
        for layer in self.enc:
            x = layer(x, pad)
        return x, pad
    def decode(self, tgt_in, enc, pad):
        z = self.tgt_emb(tgt_in) * math.sqrt(self.d) + self.pos[:tgt_in.shape[1]]
        for layer in self.dec:
            z = layer(z, enc, pad)
        return self.out(z)
    def forward(self, src, tgt_in):
        enc, pad = self.encode(src)
        return self.decode(tgt_in, enc, pad)

# The toy translation task from L6/L7 (same generator and seed).
LEX = {"the": "le", "a": "un", "dog": "kaz", "cat": "miro", "bird": "tupi", "farmer": "olan", "child": "bebi",
       "teacher": "sanu", "ball": "pilo", "book": "libu", "apple": "mazo", "letter": "kato", "big": "gros", "small": "pik",
       "red": "roj", "old": "vej", "happy": "felo", "sees": "vidas", "eats": "manjas", "finds": "trovas", "throws": "jetas",
       "likes": "amas", "in": "ene", "near": "apud", "with": "kun", "garden": "jardo", "house": "domo", "park": "parko", ".": "."}
N_ANIM = ["dog", "cat", "bird", "farmer", "child", "teacher"]; N_OBJ = ["ball", "book", "apple", "letter"] + N_ANIM
N_PLACE = ["garden", "house", "park"]; ADJS = ["big", "small", "red", "old", "happy"]
VERBS_MT = ["sees", "eats", "finds", "throws", "likes"]; PREPS = ["in", "near", "with"]
def noun_phrase(nouns, max_adj, rg):
    det = rg.choice(["the", "a"]); adjs = [rg.choice(ADJS) for _ in range(rg.choice(range(max_adj + 1)))]; n_ = rg.choice(nouns)
    return [det] + adjs + [n_], [LEX[det], LEX[n_]] + [LEX[a] for a in reversed(adjs)]
def translation_pair(rg=random, max_adj=2, max_pp=2):
    s1, t1 = noun_phrase(N_ANIM, max_adj, rg); vb = rg.choice(VERBS_MT); s2, t2 = noun_phrase(N_OBJ, max_adj, rg)
    src, tgt = s1 + [vb] + s2, t1 + t2
    for _ in range(rg.choice(range(max_pp + 1))):
        pr = rg.choice(PREPS); s3, t3 = noun_phrase(N_PLACE + N_ANIM, max_adj, rg); src += [pr] + s3; tgt += t3 + [LEX[pr]]
    return src + ["."], tgt + [LEX[vb], "."]
random.seed(0)
mt_data = [translation_pair() for _ in range(10000)]; mt_train, mt_test = mt_data[:8500], mt_data[8500:]
src_vocab = ["<pad>"] + sorted({w for s, _ in mt_data for w in s}); tgt_vocab = ["<pad>", "<s>"] + sorted({w for _, t in mt_data for w in t})
S2I = {w: i for i, w in enumerate(src_vocab)}; T2I = {w: i for i, w in enumerate(tgt_vocab)}

def pad_batch(seqs, vocab_map, prefix=None):
    L = max(len(s) for s in seqs) + (1 if prefix else 0)
    out = torch.zeros(len(seqs), L, dtype=torch.long)
    for b, s in enumerate(seqs):
        ids = ([vocab_map[prefix]] if prefix else []) + [vocab_map[w] for w in s]
        out[b, :len(ids)] = torch.tensor(ids)
    return out

@torch.no_grad()
def translate(model, src, max_len=40):
    model.eval()
    enc, pad = model.encode(pad_batch([src], S2I))
    ys = [T2I["<s>"]]
    for _ in range(max_len):
        logits = model.decode(torch.tensor([ys]), enc, pad)
        ys.append(int(logits[0, -1].argmax()))
        if tgt_vocab[ys[-1]] == ".":
            break
    return [tgt_vocab[i] for i in ys[1:]]

torch.manual_seed(0)
model = Transformer(len(src_vocab), len(tgt_vocab))
opt_t = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98))
for epoch in range(8):
    model.train()
    perm = np.random.default_rng(epoch).permutation(len(mt_train))
    for k in range(0, len(perm), 64):
        batch = [mt_train[i] for i in perm[k:k + 64]]
        src = pad_batch([s for s, _ in batch], S2I)
        tgt_in = pad_batch([t[:-1] for _, t in batch], T2I, prefix="<s>")       # shifted right
        tgt_out = pad_batch([t for _, t in batch], T2I)
        logits = model(src, tgt_in)
        loss = F.cross_entropy(logits.reshape(-1, len(tgt_vocab)), tgt_out.reshape(-1), ignore_index=0)
        opt_t.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt_t.step()
    exact = np.mean([translate(model, s) == t for s, t in mt_test[:300]])
    print(f"epoch {epoch + 1}: loss {loss.item():.4f}   exact-match on 300 test sentences {exact:.1%}")

---
## 11. Summary

**Yay, you now understand Transformers!**

| Piece | What it does / why it's there |
|---|---|
| Self-attention | $\text{softmax}(XQ(XK)^\top/\sqrt{d_k})XV$: every word looks up every other word; $O(1)$ path length, fully parallel, $O(n^2d)$ cost |
| Feed-forward | Adds per-position non-linearity (attention alone only re-averages values) |
| Residual connections | Identity path keeps information and gradients flowing through deep stacks |
| LayerNorm | Per-token normalization stabilizes the inputs each layer sees |
| $1/\sqrt{d_k}$ scaling | Keeps score variance ~1, so the softmax doesn't saturate and gradients flow |
| Positional encodings | Self-attention is permutation-equivariant; sinusoidal / learned / relative / RoPE inject order |
| Multi-head attention | $h$ independent attention patterns at the cost of one (reshape trick) |
| Causal mask | Scores to future positions set to $-\infty$: parallel training without cheating |
| Cross-attention | Decoder queries, encoder keys and values |
| Drawbacks | Quadratic cost (Linformer, BigBird, FlashAttention); absolute positions don't extrapolate |

### Exercises
1. Replace the learned position embeddings in the NumPy model with **RoPE** (apply `rope` to q and k inside `mha_forward`; the backward pass applies the inverse rotation to the gradients). Re-run §8.5: does the model now handle longer sentences?
2. Add **dropout** after attention and the FFN in the NumPy model (forward and backward), and compare dev perplexity.
3. Convert the NumPy model to **post-LN** ("Add & Norm" as in the original paper). Does it train as easily with the same learning rate?
4. Implement **SwiGLU** in place of the ReLU FFN (keep the parameter count similar) and gradient-check it.
5. Visualize all heads' attention maps over a whole sentence (not just from the verb). Can you find a "previous token" head?
6. In the PyTorch model, switch from sinusoidal to learned positions, and test on translation pairs longer than any in training (`max_pp=4`).
7. Train L5's NumPy RNN language model on `train_data` and evaluate agreement on `test_long`, split by whether the verb is beyond position `T_TRAIN`. Compare with §8.5.

### References
* Vaswani et al. 2017, *Attention Is All You Need*.
* Ba, Kiros & Hinton 2016, *Layer Normalization*; Xiong et al. 2020, *On Layer Normalization in the Transformer Architecture*.
* Shaw, Uszkoreit & Vaswani 2018, *Self-Attention with Relative Position Representations*; Su et al. 2021, *RoFormer* (RoPE); Press et al. 2022 (ALiBi).
* Kaplan et al. 2020, *Scaling Laws for Neural Language Models*.
* Wang et al. 2020 (Linformer); Zaheer et al. 2021 (BigBird); Dao et al. 2022 (FlashAttention).
* Narang et al. 2021, *Do Transformer Modifications Transfer Across Implementations and Applications?*
* Alammar, *The Illustrated Transformer* (blog); Rush, *The Annotated Transformer* (Harvard NLP); Karpathy, *nanoGPT*.

**Next:** Lecture 9, pretraining (BERT, GPT, T5) and why it changed everything.